# Lasso and Sparse Compressed Sensing

根据《机器学习数值分析》结课项目需求文档编写。本Notebook实现了多种稀疏优化算法，包括近端梯度下降法 (ISTA/FISTA)、坐标轴下降法 (CD) 和交替方向乘子法 (ADMM)，并对比了它们的收敛性，最后应用于一维压缩感知信号恢复。

## 1. 导入依赖库与数据生成
生成人工稀疏信号、高斯测量矩阵和带有噪声的观测数据。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# 设置随机种子以保证可复现性
np.random.seed(42)

# 参数设置
N = 1000  # 原始信号维度
M = 300   # 测量维度 (M < N)
S = 50    # 信号稀疏度 (非零元素个数)

# 生成真实稀疏信号 x_true
x_true = np.zeros(N)
nonzero_indices = np.random.choice(N, S, replace=False)
x_true[nonzero_indices] = np.random.randn(S)

# 生成测量矩阵 A (Gaussian Random Matrix)
A = np.random.randn(M, N) / np.sqrt(M)

# 生成观测噪声
sigma = 0.05
noise = sigma * np.random.randn(M)

# 生成观测信号 y
y = A @ x_true + noise

print(f"信号维度 N={N}, 测量维度 M={M}, 稀疏度 S={S}")


## 2. 目标函数与软阈值算子
Lasso的无约束优化目标函数为：$\min_{x} \frac{1}{2} \|Ax - y\|_2^2 + \lambda \|x\|_1$

其中 $\ell_1$ 范数的近端算子 (Proximal Operator) 为软阈值算子 (Soft Thresholding)。

In [ ]:
def f_val(x, A, y, lam):
    """计算Lasso目标函数值"""
    return 0.5 * np.linalg.norm(A @ x - y)**2 + lam * np.linalg.norm(x, 1)

def soft_threshold(v, kappa):
    """软阈值算子"""
    return np.sign(v) * np.maximum(np.abs(v) - kappa, 0.)

# 设定正则化参数 lambda
lam = 0.1 * np.max(np.abs(A.T @ y))
print(f"选择的正则化参数 lambda={lam:.4f}")


## 3. 优化算法实现

In [ ]:
def ista(A, y, lam, alpha, max_iter=500, tol=1e-5):
    """ISTA (Iterative Shrinkage-Thresholding Algorithm)"""
    x = np.zeros(A.shape[1])
    obj_vals = []
    for k in range(max_iter):
        grad = A.T @ (A @ x - y)
        x_new = soft_threshold(x - alpha * grad, alpha * lam)
        obj_vals.append(f_val(x_new, A, y, lam))
        if np.linalg.norm(x_new - x) < tol:
            break
        x = x_new
    return x, obj_vals

def fista(A, y, lam, alpha, max_iter=500, tol=1e-5):
    """FISTA (Fast ISTA)"""
    x = np.zeros(A.shape[1])
    z = np.zeros(A.shape[1])
    t = 1.0
    obj_vals = []
    for k in range(max_iter):
        grad = A.T @ (A @ z - y)
        x_new = soft_threshold(z - alpha * grad, alpha * lam)
        obj_vals.append(f_val(x_new, A, y, lam))
        if np.linalg.norm(x_new - x) < tol:
            break
        t_new = (1.0 + np.sqrt(1.0 + 4.0 * t**2)) / 2.0
        z = x_new + ((t - 1.0) / t_new) * (x_new - x)
        x = x_new
        t = t_new
    return x, obj_vals

def coordinate_descent(A, y, lam, max_iter=500, tol=1e-5):
    """Coordinate Descent (CD)"""
    N = A.shape[1]
    x = np.zeros(N)
    obj_vals = []
    r = y - A @ x
    normA_sq = np.sum(A**2, axis=0)
    
    for k in range(max_iter):
        x_old = x.copy()
        for j in range(N):
            v_j = A[:, j].T @ r + normA_sq[j] * x[j]
            x_new_j = soft_threshold(v_j, lam) / normA_sq[j]
            r -= A[:, j] * (x_new_j - x[j])
            x[j] = x_new_j
        obj_vals.append(f_val(x, A, y, lam))
        if np.linalg.norm(x - x_old) < tol:
            break
    return x, obj_vals

def admm(A, y, lam, rho=1.0, max_iter=500, tol=1e-5):
    """Alternating Direction Method of Multipliers (ADMM)"""
    N = A.shape[1]
    x = np.zeros(N)
    z = np.zeros(N)
    u = np.zeros(N)
    obj_vals = []
    
    AtA = A.T @ A
    invMat = np.linalg.inv(AtA + rho * np.eye(N))
    Aty = A.T @ y
    
    for k in range(max_iter):
        x_new = invMat @ (Aty + rho * (z - u))
        z_new = soft_threshold(x_new + u, lam / rho)
        u_new = u + x_new - z_new
        obj_vals.append(f_val(x_new, A, y, lam))
        if np.linalg.norm(x_new - x) < tol and np.linalg.norm(z_new - z) < tol:
            break
        x, z, u = x_new, z_new, u_new
    return x, obj_vals


## 4. 算法运行与收敛性对比分析

In [ ]:
# 计算Lipschitz常数，用于设定步长 alpha
L = np.linalg.norm(A, ord=2)**2
alpha = 1.0 / L

print("Running ISTA...")
x_ista, obj_ista = ista(A, y, lam, alpha)

print("Running FISTA...")
x_fista, obj_fista = fista(A, y, lam, alpha)

print("Running Coordinate Descent...")
x_cd, obj_cd = coordinate_descent(A, y, lam)

print("Running ADMM...")
x_admm, obj_admm = admm(A, y, lam, rho=1.0)

# 绘制收敛曲线
plt.figure(figsize=(10, 6))
plt.plot(obj_ista, label='ISTA', linewidth=2)
plt.plot(obj_fista, label='FISTA', linewidth=2)
plt.plot(obj_cd, label='CD (per epoch)', linewidth=2)
plt.plot(obj_admm, label='ADMM', linewidth=2)
plt.yscale('log')
plt.xlabel('Iterations / Epochs')
plt.ylabel('Objective Function Value (Log Scale)')
plt.legend()
plt.title('Convergence Comparison of Lasso Solvers')
plt.grid(True)
plt.show()


从图中可以看出，FISTA相对于ISTA具有明显的加速效果，而坐标轴下降法(CD)在处理稀疏问题时通常单步更新成本较高但收敛曲线十分陡峭，ADMM的收敛特性受参数$\rho$影响较大。

## 5. 压缩感知恢复结果可视化
对比原始信号与我们恢复出的稀疏信号。

In [ ]:
plt.figure(figsize=(15, 4))

plt.subplot(131)
plt.stem(x_true, markerfmt='k.', linefmt='k-', basefmt='k-')
plt.title('True Sparse Signal')
plt.xlabel('Index')
plt.ylabel('Amplitude')

plt.subplot(132)
plt.stem(x_fista, markerfmt='r.', linefmt='r-', basefmt='k-')
plt.title('Recovered by FISTA')
plt.xlabel('Index')

plt.subplot(133)
plt.stem(x_cd, markerfmt='b.', linefmt='b-', basefmt='k-')
plt.title('Recovered by Coordinate Descent')
plt.xlabel('Index')

plt.tight_layout()
plt.show()

# 计算均方误差 (MSE)
mse_fista = np.mean((x_fista - x_true)**2)
mse_cd = np.mean((x_cd - x_true)**2)
print(f"FISTA Recovery MSE: {mse_fista:.6f}")
print(f"CD Recovery MSE:    {mse_cd:.6f}")
